In [58]:
!pip install pandas numpy scikit-learn matplotlib seaborn xgboost
!pip install fastapi uvicorn streamlit pyngrok

In [59]:
import pandas as pd

df = pd.read_csv('/content/rideflow_datasets.csv')
df.head()

,ride_id,timestamp,pickup_zone,drop_zone,pickup_lat,pickup_long,drop_lat,drop_long,driver_id,customer_id,...,surge_multiplier,driver_rating,customer_rating,estimated_eta_min,actual_eta_min,ride_status,traffic_level,weather,driver_active,feedback_text
0,95.247911,2025-01-02 01:30:00,Anna Nagar,Adyar,12.880239,80.148410,13.028939,80.163941,1842.701958,6072.494896,...,1.001779,4.350624,4.037232,11.778023,18.304775,cancelled,low,clear,-0.036560,Driver was polite
1,439.187632,2025-01-05 12:45:00,T Nagar,Tambaram,13.092441,80.165458,13.142711,80.149376,1186.296422,5942.228896,...,1.193147,4.524196,3.324278,4.430894,13.343961,completed,low,cloudy,0.988999,Driver cancelled suddenly
2,876.685389,2025-01-09 23:00:00,Anna Nagar,Tambaram,12.817965,80.161839,12.943527,80.166040,1297.199801,5829.181415,...,2.008478,4.054085,4.979153,19.202891,12.039878,completed,low,rain,0.005750,Driver cancelled suddenly
3,275.337197,2025-01-03 19:30:00,T Nagar,Velachery,13.125103,80.143306,13.209127,80.126008,1765.474261,5429.619496,...,1.218528,3.689937,3.099466,18.711931,7.535792,completed,low,clear,1.023604,Good experience
4,106.743950,2025-01-02 02:30:00,Tambaram,Tambaram,13.143513,80.302596,13.078330,80.189672,1565.653849,5079.081677,...,1.497370,3.545512,3.073704,10.786351,12.104096,completed,high,cloudy,1.016716,Vehicle was not clean


In [60]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
df['weekday'] = df['timestamp'].dt.weekday

df.fillna(method='ffill', inplace=True)

/tmp/ipykernel_35711/262909592.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


In [61]:
#Ride Demand Prediction (MODEL 1)
demand = df.groupby(['pickup_zone','hour']).size().reset_index(name='ride_count')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X = demand[['hour']]
y = demand['ride_count']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

model_demand = RandomForestRegressor()
model_demand.fit(X_train,y_train)

RandomForestRegressor()

In [62]:
#Driver Supply Prediction
supply = df.groupby(['pickup_zone','hour'])['driver_id'].nunique().reset_index(name='drivers')

X = supply[['hour']]
y = supply['drivers']

model_supply = RandomForestRegressor()
model_supply.fit(X,y)

RandomForestRegressor()

In [63]:
#Dynamic Pricing
def surge_pricing(demand, supply):
    ratio = demand / max(supply,1)
    surge = min(max(ratio,1),3)
    return surge

In [64]:
#Cancellation Prediction

from sklearn.linear_model import LogisticRegression
import pandas as pd

features = ['fare_price','traffic_level','surge_multiplier']

X = df[features]
y = df['ride_status']  # 0 or 1

# Convert 'traffic_level' to numerical using one-hot encoding
X = pd.get_dummies(X, columns=['traffic_level'], drop_first=True)

# Convert target variable 'ride_status' to numerical (0 for cancelled, 1 for completed)
y = y.apply(lambda x: 0 if x == 'cancelled' else 1)

model_cancel = LogisticRegression()
model_cancel.fit(X,y)

LogisticRegression()

In [65]:
#ETA Prediction

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(50, input_shape=(10,3)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [66]:
#Build FastAPI

from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "RideFlow API running"}

@app.get("/predict")
def predict(hour:int):
    demand = model_demand.predict([[hour]])[0]
    supply = model_supply.predict([[hour]])[0]
    surge = surge_pricing(demand,supply)

    return {
        "demand": demand,
        "supply": supply,
        "surge": surge
    }

In [72]:
import pickle

with open("demand.pkl", "wb") as f:
    pickle.dump(model_demand, f)

with open("supply.pkl", "wb") as f:
    pickle.dump(model_supply, f)

with open("cancel.pkl", "wb") as f:
    pickle.dump(model_cancel, f)